In [ ]:
import os
import io

import numpy as np
import pyarrow.parquet as pq
from PIL import Image
from tqdm.auto import tqdm

import torch
from torchvision import transforms
from transformers import RobertaTokenizer

from qdrant_client import QdrantClient
from qdrant_client.http import models

from models.scold.model import LVL


# -------------------------
# Config
# -------------------------
PARQUET_PATH = "data/plantwild/plantwild_sampled.parquet"
QDRANT_URL = os.getenv("QDRANT_URL", "http://localhost:6334")
COLLECTION_NAME = "leaf_disease_collection"

# Tuning knobs (start here; adjust if needed)
READ_BATCH_ROWS = 256          # how many rows to read + embed at once
IMAGE_MICROBATCH = 16          # image forward-pass microbatch to avoid GPU OOM
UPLOAD_BATCH_SIZE = 2048       # Qdrant suggests 1k–10k for 100k–1M scale
UPLOAD_PARALLEL = 4            # set to number of CPU cores you want to use


# -------------------------
# Model / encoding
# -------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LVL()
model.load_state_dict(torch.load("models/scold/scold.pt", map_location=device))
model.to(device)
model.eval()

tokenizer = RobertaTokenizer.from_pretrained("roberta-base")

transform = transforms.Compose(
    [
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ]
)


def encode_text(texts: list[str]) -> np.ndarray:
    """Returns float32 array of shape (B, 512)."""
    inputs = tokenizer(texts, return_tensors="pt", padding=True, truncation=True).to(device)
    with torch.no_grad():
        emb = model.get_texts_feature(inputs["input_ids"], inputs["attention_mask"])
    return emb.detach().cpu().numpy().astype(np.float32, copy=False)


def encode_images_from_bytes(images_bytes: list[bytes], microbatch: int = IMAGE_MICROBATCH) -> np.ndarray:
    """Returns float32 array of shape (B, 512)."""
    out = []
    for j in range(0, len(images_bytes), microbatch):
        chunk = images_bytes[j : j + microbatch]
        imgs = []
        for b in chunk:
            img = Image.open(io.BytesIO(b)).convert("RGB")
            imgs.append(transform(img))
        batch = torch.stack(imgs, dim=0).to(device)

        with torch.no_grad():
            emb = model.get_images_features(batch)

        out.append(emb.detach().cpu().numpy())

    return np.concatenate(out, axis=0).astype(np.float32, copy=False)


# -------------------------
# Qdrant collection setup (no recreate_collection)
# -------------------------
def reset_collection(client: QdrantClient, name: str) -> None:
    if client.collection_exists(name):
        client.delete_collection(name)

    client.create_collection(
        collection_name=name,
        vectors_config={
            "text": models.VectorParams(size=512, distance=models.Distance.COSINE, on_disk=True),
            "image": models.VectorParams(size=512, distance=models.Distance.COSINE, on_disk=True),
        },
    )


# -------------------------
# Parquet streaming -> PointStruct iterator
# -------------------------
def iter_points_from_parquet(parquet_path: str):
    pf = pq.ParquetFile(parquet_path)
    total = pf.metadata.num_rows if pf.metadata else None

    pbar = tqdm(total=total, desc="Preparing points", unit="pt")
    point_id = 0

    try:
        for record_batch in pf.iter_batches(batch_size=READ_BATCH_ROWS, columns=["caption", "label", "image"]):
            df = record_batch.to_pandas()

            captions = [str(x) for x in df["caption"].tolist()]

            # image column in your dataset looks like a struct/dict containing {"bytes": ...}
            img_bytes_list = []
            for x in df["image"].tolist():
                if isinstance(x, dict) and "bytes" in x:
                    img_bytes_list.append(x["bytes"])
                else:
                    # if it is already raw bytes
                    img_bytes_list.append(x)

            text_vecs = encode_text(captions)                    # (B, 512)
            img_vecs = encode_images_from_bytes(img_bytes_list)  # (B, 512)

            labels = df["label"].tolist()

            for i in range(len(df)):
                payload = {
                    "caption": captions[i],
                    "label": str(labels[i]),
                    "source_id": int(point_id),
                }

                yield models.PointStruct(
                    id=int(point_id),
                    vector={
                        "text": text_vecs[i].tolist(),
                        "image": img_vecs[i].tolist(),
                    },
                    payload=payload,
                )

                point_id += 1
                pbar.update(1)
    finally:
        pbar.close()









In [ ]:
client = QdrantClient(url=QDRANT_URL, prefer_grpc=True)

reset_collection(client, COLLECTION_NAME)

points_iter = iter_points_from_parquet(PARQUET_PATH)

# Recommended for your scale: upload_points with batching + parallelism
# (Qdrant course suggests batch sizes ~1k–10k for 100k–1M). :contentReference[oaicite:3]{index=3}
client.upload_points(
    collection_name=COLLECTION_NAME,
    points=points_iter,
    batch_size=UPLOAD_BATCH_SIZE,
    parallel=UPLOAD_PARALLEL,
    wait=False,
)

print("Done.")